# 🧪 Laboratorio 7: Clasificación con Datos Financieros — Soluciones

**Módulo 03** | **Sesión 5** | **Duración estimada: 45min**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-03-modelado-predictivo/laboratorios/lab_07_clasificacion_soluciones.ipynb)

---

Este notebook contiene las **soluciones completas** con código, interpretaciones y comentarios didácticos.

## 🎯 Objetivos de Aprendizaje

| # | Competencia | Descripción | Nivel |
|---|-------------|-------------|-------|
| 1 | Ingeniería de features | Crear variables derivadas (ratios financieros) a partir de datos crudos | Aplicación |
| 2 | Clasificación múltiple | Entrenar y comparar 3 modelos de clasificación diferentes | Análisis |
| 3 | Evaluación comparativa | Comparar modelos usando métricas estándar de clasificación | Evaluación |
| 4 | Selección de modelo | Elegir el mejor modelo y justificar la elección | Evaluación |

## 💡 ¿Por qué es importante para tu práctica docente?

En las carreras de Contaduría, Administración y Economía de FACES, el **análisis de estados financieros** es una competencia fundamental. En este laboratorio aplicarás técnicas de machine learning para clasificar empresas como "rentables" o "no rentables" — un problema real que enfrentan analistas financieros, auditores y asesores empresariales.

Al comparar múltiples modelos, desarrollarás el criterio para elegir la herramienta adecuada según el contexto.

In [ ]:
# Librerías necesarias — ejecuta esta celda primero
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Librerías cargadas correctamente ✓')

---

## Ejercicio 1: Crear la variable objetivo

In [ ]:
# ============================================================
# SOLUCIÓN Ejercicio 1: Crear la variable objetivo
# ============================================================

# 1. Cargar el CSV
fin = pd.read_csv('../datasets/empresarial/estados_financieros.csv')

# 2. Explorar los datos
print(f'Dimensiones: {fin.shape[0]:,} filas x {fin.shape[1]} columnas')
print(f'\nColumnas disponibles: {list(fin.columns)}')
print('\nPrimeras 5 filas:')
fin.head()

# 3. Crear la variable objetivo: rentable = 1 si utilidad_neta > 0
fin['rentable'] = (fin['utilidad_neta_musd'] > 0).astype(int)

# 4. Distribución de la variable objetivo
print('\nDistribución de la variable objetivo:')
print(fin['rentable'].value_counts())
print(f'\nPorcentaje de empresas rentables: {fin["rentable"].mean()*100:.1f}%')

# NOTA: Si todas las empresas son rentables (100%), la clasificación
# no tendría sentido. En ese caso usamos "alto riesgo financiero" como
# variable alternativa (endeudamiento > 0.55).
# Verificamos:
if fin['rentable'].nunique() < 2:
    print('\n⚠ Todas las empresas son rentables. Usamos variable alternativa:')
    print('  alto_riesgo = endeudamiento > 0.55')
    fin['endeudamiento'] = fin['pasivos_musd'] / fin['activos_musd']
    fin['rentable'] = (fin['endeudamiento'] > 0.55).astype(int)
    print(fin['rentable'].value_counts().rename({0: 'Bajo riesgo (0)', 1: 'Alto riesgo (1)'}))
    print(f'  {fin["rentable"].mean()*100:.1f}% alto riesgo')

---

## Ejercicio 2: Calcular ratios financieros

In [ ]:
# ============================================================
# SOLUCIÓN Ejercicio 2: Calcular ratios financieros
# ============================================================

# 1. ROA = utilidad_neta / activos (eficiencia en uso de activos)
fin['ROA'] = fin['utilidad_neta_musd'] / fin['activos_musd']

# 2. Endeudamiento (ya calculado si usamos variable alternativa)
if 'endeudamiento' not in fin.columns:
    fin['endeudamiento'] = fin['pasivos_musd'] / fin['activos_musd']

# 3. Margen neto = utilidad_neta / ingresos (rentabilidad por dólar de ingreso)
fin['margen_neto'] = fin['utilidad_neta_musd'] / fin['ingresos_musd']

# 4. Ratios adicionales útiles
fin['rotacion_activos'] = fin['ingresos_musd'] / fin['activos_musd']
fin['ratio_costos'] = fin['costos_musd'] / fin['ingresos_musd']

# Eliminar filas con valores infinitos o NaN después del cálculo
fin.replace([np.inf, -np.inf], np.nan, inplace=True)
fin.dropna(inplace=True)

print(f'Registros después de limpieza: {len(fin):,}')

# Estadísticas descriptivas de los ratios
print('\nEstadísticas de los ratios financieros:')
fin[['ROA', 'endeudamiento', 'margen_neto', 'rotacion_activos', 'ratio_costos']].describe().round(4)

---

## Ejercicio 3: Regresión Logística

In [ ]:
# ============================================================
# SOLUCIÓN Ejercicio 3: Regresión Logística
# ============================================================

# 1. Definir features y target
#    No incluimos endeudamiento si se usó para crear la variable objetivo
#    Usamos ratios financieros que no contengan la variable objetivo directamente
feature_cols = ['margen_neto', 'rotacion_activos', 'ratio_costos', 'activos_musd']
X = fin[feature_cols]
y = fin['rentable']

# 2. Dividir en train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Clases en train: {y_train.value_counts().to_dict()}')

# 3. Entrenar Regresión Logística
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)

# 4. Generar predicciones
y_pred_lr = log_reg.predict(X_test)

# 5. Matriz de confusión como heatmap
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred_lr)
labels = ['Bajo riesgo', 'Alto riesgo']
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel('Predicción')
ax.set_ylabel('Valor real')
ax.set_title('Matriz de Confusión — Regresión Logística')
plt.tight_layout()
plt.show()

# 6. Classification report
print('\nReporte de clasificación — Regresión Logística:')
print(classification_report(y_test, y_pred_lr, target_names=labels))

# Guardar accuracy para comparación posterior
acc_lr = accuracy_score(y_test, y_pred_lr)
print(f'Accuracy: {acc_lr:.4f}')

---

## Ejercicio 4: Árbol de Decisión

In [ ]:
# ============================================================
# SOLUCIÓN Ejercicio 4: Árbol de Decisión
# ============================================================

# 1. Crear el clasificador con profundidad máxima = 3
arbol = DecisionTreeClassifier(max_depth=3, random_state=42)

# 2. Entrenar con los datos de train
arbol.fit(X_train, y_train)

# 3. Generar predicciones sobre test
y_pred_dt = arbol.predict(X_test)

# 4. Visualizar el árbol de decisión
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(arbol, feature_names=feature_cols,
          class_names=['Bajo riesgo', 'Alto riesgo'],
          filled=True, rounded=True, fontsize=10, ax=ax)
ax.set_title('Árbol de Decisión — Clasificación de Riesgo Financiero', fontsize=16)
plt.tight_layout()
plt.show()

# 5. Accuracy del modelo
acc_dt = accuracy_score(y_test, y_pred_dt)
print(f'\nAccuracy del Árbol de Decisión: {acc_dt:.4f}')

---

## Ejercicio 5: Random Forest

In [ ]:
# ============================================================
# SOLUCIÓN Ejercicio 5: Random Forest
# ============================================================

# 1. Crear el clasificador con 100 árboles
rf = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. Entrenar con los datos de train
rf.fit(X_train, y_train)

# 3. Generar predicciones sobre test
y_pred_rf = rf.predict(X_test)

# 4. Accuracy del Random Forest
acc_rf = accuracy_score(y_test, y_pred_rf)
print(f'📊 Accuracy del Random Forest: {acc_rf:.4f}')

# 5. Tabla comparativa de los 3 modelos
comparacion = pd.DataFrame({
    'Modelo': ['Regresión Logística', 'Árbol de Decisión', 'Random Forest'],
    'Accuracy': [acc_lr, acc_dt, acc_rf]
})
comparacion['Accuracy'] = comparacion['Accuracy'].round(4)
comparacion = comparacion.sort_values('Accuracy', ascending=False)

print('\n📊 Comparación de modelos:')
display(comparacion)

# Identificar el mejor
mejor = comparacion.iloc[0]
print(f'\n📌 El mejor modelo es: {mejor["Modelo"]} con accuracy = {mejor["Accuracy"]:.4f}')

---

## Ejercicio 6: Selección del mejor modelo

In [ ]:
# ============================================================
# SOLUCIÓN Ejercicio 6: Selección del mejor modelo
# ============================================================

# 1. ¿Cuál modelo tiene la mejor accuracy?
print(f'📊 Mejor accuracy en test: {mejor["Modelo"]} ({mejor["Accuracy"]:.4f})')

# 2. Importancia de variables del Random Forest
importancias = pd.DataFrame({
    'Feature': feature_cols,
    'Importancia': rf.feature_importances_
}).sort_values('Importancia', ascending=True)  # ascendente para barh

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(importancias['Feature'], importancias['Importancia'], color='steelblue')
ax.set_xlabel('Importancia')
ax.set_title('Importancia de Variables — Random Forest')
plt.tight_layout()
plt.show()

# 3. Feature más importante
top_feat = importancias.iloc[-1]  # última fila = mayor importancia
print(f'\n📌 Feature más importante: {top_feat["Feature"]} ({top_feat["Importancia"]:.4f})')

# 4. Justificación financiera
print(f'\n📌 Justificación financiera:')
print(f'  - La variable más importante tiene sentido porque determina directamente')
print(f'    la capacidad de la empresa para generar utilidades.')
print(f'  - El endeudamiento indica el nivel de riesgo financiero.')
print(f'  - La relación ingresos-costos determina el margen operativo.')

# 5. Recomendación como analista de crédito
print(f'\n📌 Recomendación como analista de crédito:')
print(f'  - Para análisis de crédito, el Árbol de Decisión es valioso por su')
print(f'    interpretabilidad: se puede explicar exactamente por qué se aprobó')
print(f'    o rechazó un crédito, lo cual es un requisito regulatorio.')
print(f'  - El Random Forest ofrece mejor accuracy pero es una "caja negra".')
print(f'  - La elección depende del contexto: si la regulación exige')
print(f'    explicabilidad, el árbol es mejor. Si prima la precisión, el RF.')

---

## 📋 Resumen

En este laboratorio practicaste:

| Paso | Qué hiciste |
|------|-------------|
| 1. Variable objetivo | Creaste una variable binaria (rentable/no rentable) |
| 2. Feature engineering | Calculaste ratios financieros como variables predictoras |
| 3. Modelo 1 | Regresión Logística con matriz de confusión |
| 4. Modelo 2 | Árbol de Decisión visualizado |
| 5. Modelo 3 | Random Forest con importancia de variables |
| 6. Comparación | Selección del mejor modelo con justificación |

## 📚 Referencias

1. Scikit-learn developers. (2024). *Supervised learning*. https://scikit-learn.org/stable/supervised_learning.html

2. Ross, S. A., Westerfield, R. W., & Jordan, B. D. (2019). *Fundamentals of Corporate Finance* (12th ed.). McGraw-Hill.

---

*Notebook desarrollado para el programa de Formación Docente en Ciencia de Datos y Business Intelligence — FACES, Universidad Catolica Andres Bello (UCAB).*